In [1]:
import pandas as pd
import numpy as np

df_feat = pd.read_pickle("dataset_features_v3.pkl")
rank_df = df_feat.copy()
rank_df.shape

(100000, 116)

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
jd_skill_text = """
embeddings retrieval ranking recommendation systems
hybrid search vector databases
sentence-transformers BGE E5 FAISS Pinecone Weaviate Qdrant OpenSearch Elasticsearch
Python evaluation frameworks NDCG MAP MRR
"""

jd_career_text = """
product company experience
production ML systems
retrieval ranking recommendation search systems
evaluation frameworks
A/B testing
NDCG MAP MRR
Python engineering
hands-on coding
shipper mindset
"""

jd_profile_text = """
senior AI engineer
6 to 8 years experience
active candidate
open to relocation
startup mindset
fast moving engineering culture
hands-on coding
scalable systems
"""

In [4]:
jd_skill_embedding = model.encode(
    jd_skill_text,
    normalize_embeddings=True
)

jd_career_embedding = model.encode(
    jd_career_text,
    normalize_embeddings=True
)

jd_profile_embedding = model.encode(
    jd_profile_text,
    normalize_embeddings=True
)

In [5]:
skill_embeddings = model.encode(
    rank_df['skills_text'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=128
)
career_embeddings = model.encode(
    rank_df['career_text'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=128
)
profile_embeddings = model.encode(
    rank_df['profile_text'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=128
)

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

In [6]:
skill_similarity = cosine_similarity(
    skill_embeddings,
    jd_skill_embedding.reshape(1,-1)
).flatten()

career_similarity = cosine_similarity(
    career_embeddings,
    jd_career_embedding.reshape(1,-1)
).flatten()

profile_similarity = cosine_similarity(
    profile_embeddings,
    jd_profile_embedding.reshape(1,-1)
).flatten()

In [7]:
rank_df['semantic_score'] = (
      0.70*career_similarity
    + 0.25*skill_similarity
    + 0.05*profile_similarity
)

In [8]:
scaler = MinMaxScaler()

rank_df['semantic_score_norm'] = scaler.fit_transform(
    rank_df[['semantic_score']]
)

In [9]:
import pickle

with open("skill_embeddings.pkl","wb") as f:
    pickle.dump(skill_embeddings,f)

with open("career_embeddings.pkl","wb") as f:
    pickle.dump(career_embeddings,f)

with open("profile_embeddings.pkl","wb") as f:
    pickle.dump(profile_embeddings,f)

In [10]:
career_cols = [
    'profile_years_of_experience',
    'current_company_tenure_months',
    'avg_job_duration_months',
    'highest_degree_score'
]

rank_df['career_score'] = scaler.fit_transform(
    rank_df[career_cols]
    .mean(axis=1)
    .values.reshape(-1,1)
)

In [11]:
behavior_cols = [
    'signal_recruiter_response_rate',
    'signal_interview_completion_rate',
    'signal_github_activity_score',
    'signal_saved_by_recruiters_30d',
    'signal_profile_views_received_30d'
]

rank_df[behavior_cols] = scaler.fit_transform(
    rank_df[behavior_cols]
)

rank_df['behavior_score'] = (
    rank_df[behavior_cols]
    .mean(axis=1)
)

In [12]:
availability_cols = [
    'signal_open_to_work_flag',
    'signal_willing_to_relocate'
]
rank_df['signal_last_active_date'] = pd.to_datetime(
    rank_df['signal_last_active_date']
)

today = pd.Timestamp.today()

rank_df['days_since_last_active'] = (
    today - rank_df['signal_last_active_date']
).dt.days

rank_df['recent_activity_score'] = 1 - MinMaxScaler().fit_transform(
    rank_df[['days_since_last_active']]
)

rank_df['availability_score'] = (
    0.4*rank_df['signal_open_to_work_flag']
    +0.2*rank_df['signal_willing_to_relocate']
    +0.4*rank_df['recent_activity_score']
)

In [13]:
skill_cols = [
    'num_skills',
    'avg_skill_duration_months',
    'avg_skill_endorsements'
]
rank_df[skill_cols] = scaler.fit_transform(
    rank_df[skill_cols]
)

rank_df['skill_score'] = (
    rank_df[skill_cols]
    .mean(axis=1)
)

In [31]:
strong_positive = [
    'Recommendation Systems Engineer',
    'ML Engineer',
    'AI Research Engineer',
    'AI Specialist',
    'Backend Engineer',
    'Software Engineer',
    'Senior Software Engineer',
    'Data Scientist',
    'Data Engineer',
    'Senior Data Engineer',
    'Analytics Engineer',
    'Cloud Engineer',
    'DevOps Engineer',
    'Machine Learning Engineer',
    'Senior Machine Learning Engineer',
    'Staff Machine Learning Engineer',
    'Applied ML Engineer',
    'AI Engineer',
    'Senior AI Engineer',
    'NLP Engineer'
]
mild_positive = [
    '.NET Developer',
    'Java Developer',
    'Frontend Engineer',
    'Full Stack Developer',
    'Mobile Developer'
]
neutral = [
    'Business Analyst',
    'QA Engineer',
    'Project Manager',
    'Data Analyst'
]
strong_negative = [
    'HR Manager',
    'Marketing Manager',
    'Content Writer',
    'Graphic Designer',
    'Customer Support',
    'Sales Executive',
    'Accountant'
]
mild_negative = [
    'Mechanical Engineer',
    'Civil Engineer',
    'Operations Manager'
]

In [32]:
def title_prior(title):
    
    if title in strong_positive:
        return 1

    elif title in mild_positive:
        return 0.5

    elif title in strong_negative:
        return -1

    elif title in mild_negative:
        return -0.5

    else:
        return 0

rank_df['title_prior'] = (
    rank_df['profile_current_title']
    .apply(title_prior)
)

In [33]:
non_ai_titles = [
    'HR Manager',
    'Marketing Manager',
    'Content Writer',
    'Graphic Designer',
    'Customer Support',
    'Sales Executive',
    'Accountant'
]

ai_words = [
    'rag',
    'langchain',
    'pinecone',
    'embeddings',
    'vector search',
    'genai',
    'llm',
    'openai'
]

In [34]:
def transition_penalty(row):

    title = str(row['profile_current_title']).lower()
    profile = str(row['profile_text']).lower()

    if (
        any(t.lower() == title for t in non_ai_titles)
        and
        any(word in profile for word in ai_words)
    ):
        return -1

    return 0

rank_df['transition_penalty'] = (
    rank_df.apply(
        transition_penalty,
        axis=1
    )
)

In [35]:
rank_df['job_switch_penalty'] = (
    rank_df['job_switch_count'] > 8
).astype(int)
rank_df['inactive_penalty'] = (
    rank_df['days_since_last_active'] > 180
).astype(int)

In [36]:
rank_df['integrity_penalty'] = (
    rank_df['job_switch_penalty']
    +
    rank_df['inactive_penalty']
)
from sklearn.preprocessing import MinMaxScaler

rank_df['integrity_penalty'] = (
    MinMaxScaler()
    .fit_transform(
        rank_df[['integrity_penalty']]
    )
)

In [22]:
keywords = [
    'recommendation',
    'retrieval',
    'ranking',
    'semantic search',
    'information retrieval',
    'vector search',
    'faiss',
    'bm25'
]
def retrieval_bonus(row):
    text = (
        str(row['skills_text']) +
        " " +
        str(row['career_text'])
    ).lower()

    return int(
        any(word in text for word in keywords)
    )

In [23]:
rank_df['retrieval_bonus'] = (
    rank_df.apply(
        retrieval_bonus,
        axis=1
    )
)

In [37]:
product_industries = [
    'Software',
    'Food Delivery',
    'Fintech',
    'E-commerce',
    'EdTech',
    'SaaS',
    'AI/ML',
    'HealthTech',
    'Gaming',
    'Conversational AI',
    'AdTech',
    'HealthTech AI'
]
negative_industries = [
    'IT Services',
    'Manufacturing',
    'Consulting'
]

In [38]:
def industry_bonus(industry):
    if industry in product_industries:
        return 1
    elif industry in negative_industries:
        return -1
    return 0
rank_df['industry_bonus'] = (
    rank_df['profile_current_industry']
    .apply(industry_bonus)
)

In [39]:
service_companies = [
    'TCS',
    'Infosys',
    'Wipro',
    'Accenture',
    'Cognizant',
    'Capgemini'
]
def career_company_penalty(career_history):

    companies = [
        str(job.get('company', ''))
        for job in career_history
    ]

    if len(companies) == 0:
        return 0

    count_service = sum(
        any(
            s.lower() in company.lower()
            for s in service_companies
        )
        for company in companies
    )

    # Entire career in service companies
    if count_service == len(companies):
        return 1
    return 0

rank_df['career_company_penalty'] = (
    rank_df['career_history']
    .apply(career_company_penalty)
)

In [40]:
rank_df.loc[
    rank_df['signal_notice_period_days'] > 90,
    'notice_penalty'
] = 2.0
rank_df.loc[
    (rank_df['signal_notice_period_days'] > 30)
    &
    (rank_df['signal_notice_period_days'] <= 90),
    'notice_penalty'
] = 1.0

In [41]:
ir_words = [
    'recommendation',
    'retrieval',
    'ranking',
    'semantic search',
    'information retrieval',
    'vector search',
    'faiss',
    'bm25',
    'sentence transformers',
    'pinecone',
    'qdrant',
    'weaviate',
    'elasticsearch',
    'opensearch'
]
def ir_bonus(row):
    
    text = (
        str(row['career_text']) + " " +
        str(row['skills_text'])
    ).lower()
    
    return int(
        any(word in text for word in ir_words)
    )
rank_df['ir_bonus'] = (
    rank_df.apply(
        ir_bonus,
        axis=1
    )
)

In [42]:
rank_df['final_score'] = (
      0.40 * rank_df['semantic_score_norm']
    + 0.20 * rank_df['career_score']
    + 0.15 * rank_df['behavior_score']
    + 0.10 * rank_df['availability_score']
    + 0.10 * rank_df['skill_score']
    - 0.05 * rank_df['integrity_penalty']
    + 0.10 * rank_df['title_prior']
    + 0.10 * rank_df['transition_penalty']
    + 0.05 * rank_df['industry_bonus']
    - 0.05 * rank_df['career_company_penalty']
    - 0.05 * rank_df['notice_penalty']
    + 0.05 * rank_df['ir_bonus']
)

In [43]:
top50 = (
    rank_df[
        ['candidate_id',
         'profile_current_title',
         'final_score']
    ]
    .sort_values('final_score', ascending=False)
    .head(50)
)

print(top50)

       candidate_id             profile_current_title  final_score
41668  CAND_0041669   Recommendation Systems Engineer     0.795794
42099  CAND_0042100         Machine Learning Engineer     0.795276
77336  CAND_0077337   Staff Machine Learning Engineer     0.786282
39382  CAND_0039383               Applied ML Engineer     0.772775
66375  CAND_0066376               Applied ML Engineer     0.768286
55991  CAND_0055992                       AI Engineer     0.767136
6566   CAND_0006567                Senior AI Engineer     0.766685
6556   CAND_0006557                      NLP Engineer     0.765122
26531  CAND_0026532   Recommendation Systems Engineer     0.762294
7459   CAND_0007460                       AI Engineer     0.762281
46524  CAND_0046525  Senior Machine Learning Engineer     0.761071
75573  CAND_0075574         Machine Learning Engineer     0.760157
11161  CAND_0011162   Recommendation Systems Engineer     0.759353
50875  CAND_0050876               Applied ML Engineer     0.75

In [45]:
top100 = (
    rank_df
    .sort_values(
        'final_score',
        ascending=False
    )
    .head(100)
    .copy()
)

top100['rank'] = range(1,101)
print(top100[['rank','candidate_id','profile_current_title','final_score']])

       rank  candidate_id            profile_current_title  final_score
41668     1  CAND_0041669  Recommendation Systems Engineer     0.795794
42099     2  CAND_0042100        Machine Learning Engineer     0.795276
77336     3  CAND_0077337  Staff Machine Learning Engineer     0.786282
39382     4  CAND_0039383              Applied ML Engineer     0.772775
66375     5  CAND_0066376              Applied ML Engineer     0.768286
...     ...           ...                              ...          ...
21826    96  CAND_0021827             AI Research Engineer     0.651468
71032    97  CAND_0071033                   Data Scientist     0.650465
15577    98  CAND_0015578                      AI Engineer     0.650023
19479    99  CAND_0019480                     NLP Engineer     0.649642
49895   100  CAND_0049896                  Search Engineer     0.649284

[100 rows x 4 columns]


In [46]:
pd.set_option('display.max_rows', 100)

print(
    top100[
        ['rank',
         'candidate_id',
         'profile_current_title',
         'final_score']
    ]
)

       rank  candidate_id             profile_current_title  final_score
41668     1  CAND_0041669   Recommendation Systems Engineer     0.795794
42099     2  CAND_0042100         Machine Learning Engineer     0.795276
77336     3  CAND_0077337   Staff Machine Learning Engineer     0.786282
39382     4  CAND_0039383               Applied ML Engineer     0.772775
66375     5  CAND_0066376               Applied ML Engineer     0.768286
55991     6  CAND_0055992                       AI Engineer     0.767136
6566      7  CAND_0006567                Senior AI Engineer     0.766685
6556      8  CAND_0006557                      NLP Engineer     0.765122
26531     9  CAND_0026532   Recommendation Systems Engineer     0.762294
7459     10  CAND_0007460                       AI Engineer     0.762281
46524    11  CAND_0046525  Senior Machine Learning Engineer     0.761071
75573    12  CAND_0075574         Machine Learning Engineer     0.760157
11161    13  CAND_0011162   Recommendation Systems 

In [47]:
top100[
    ['rank',
     'candidate_id',
     'profile_current_title',
     'final_score']
].to_csv(
    "top100_preview.csv",
    index=False
)

In [59]:
def extract_keywords(row):

    text = (
        str(row['skills_text']) + " " +
        str(row['career_text'])
    ).lower()

    keyword_map = {
        'recommendation systems': 'Recommendation Systems',
        'semantic search': 'Semantic Search',
        'information retrieval': 'Information Retrieval',
        'vector search': 'Vector Search',
        'faiss': 'FAISS',
        'pinecone': 'Pinecone',
        'qdrant': 'Qdrant',
        'weaviate': 'Weaviate',
        'elasticsearch': 'Elasticsearch',
        'bm25': 'BM25',
        'sentence transformers': 'Sentence Transformers',
        'llm': 'LLMs',
        'fine-tuning': 'Fine-tuning',
        'mlops': 'MLOps',
        'langchain': 'LangChain',
        'rag': 'RAG',
        'nlp': 'NLP'
    }

    found = []

    for k, v in keyword_map.items():
        if k in text:
            found.append(v)

    return found[:3]


def generate_reason(row):

    reasons = []

    title = row['profile_current_title']
    yoe = round(row['profile_years_of_experience'], 1)
    industry = row['profile_current_industry']

    reasons.append(
        f"{title} with {yoe} years of experience in {industry}"
    )

    # Skill / career evidence
    kws = extract_keywords(row)

    if len(kws) > 0:
        reasons.append(
            "profile shows exposure to " + ", ".join(kws)
        )

    # Engagement
    if row['signal_recruiter_response_rate'] > 0.8:
        reasons.append(
            "high recruiter responsiveness suggests active engagement"
        )

    # Github activity
    if row['signal_github_activity_score'] > 0.6:
        reasons.append(
            "strong external activity signal"
        )

    # Notice period
    notice_days = int(row['signal_notice_period_days'])

    if notice_days > 90:
        reasons.append(
            f"{notice_days}-day notice period is a concern"
        )

    elif notice_days > 30:
        reasons.append(
            f"{notice_days}-day notice period raises the bar slightly"
        )

    # Rank consistency
    rank = row['rank']

    if rank <= 20:
        reasons.append(
            "overall profile aligns strongly with the role requirements"
        )

    elif rank >= 80:
        reasons.append(
            "overall alignment is weaker than higher-ranked profiles"
        )

    return ". ".join(reasons) + "."

In [60]:
top100['reasoning'] = top100.apply(generate_reason, axis=1)

In [61]:
top100['score'] = top100['final_score'].round(6)

In [62]:
submission = top100[
    [
        'candidate_id',
        'rank',
        'score',
        'reasoning'
    ]
]
submission.to_csv(
    "submission.csv",
    index=False
)

In [63]:
submission.head(10)

,candidate_id,rank,score,reasoning
41668,CAND_0041669,1,0.795794,Recommendation Systems Engineer with 8.0 years...
42099,CAND_0042100,2,0.795276,Machine Learning Engineer with 7.3 years of ex...
77336,CAND_0077337,3,0.786282,Staff Machine Learning Engineer with 7.0 years...
39382,CAND_0039383,4,0.772775,Applied ML Engineer with 7.1 years of experien...
66375,CAND_0066376,5,0.768286,Applied ML Engineer with 5.7 years of experien...
55991,CAND_0055992,6,0.767136,AI Engineer with 16.9 years of experience in F...
6566,CAND_0006567,7,0.766685,Senior AI Engineer with 7.9 years of experienc...
6556,CAND_0006557,8,0.765122,NLP Engineer with 7.9 years of experience in F...
26531,CAND_0026532,9,0.762294,Recommendation Systems Engineer with 4.8 years...
7459,CAND_0007460,10,0.762281,AI Engineer with 4.7 years of experience in So...


In [53]:
top100.columns.tolist()

['candidate_id',
 'career_history',
 'education',
 'skills',
 'certifications',
 'languages',
 'signal_profile_completeness_score',
 'signal_signup_date',
 'signal_last_active_date',
 'signal_open_to_work_flag',
 'signal_profile_views_received_30d',
 'signal_applications_submitted_30d',
 'signal_recruiter_response_rate',
 'signal_avg_response_time_hours',
 'signal_connection_count',
 'signal_endorsements_received',
 'signal_notice_period_days',
 'signal_preferred_work_mode',
 'signal_willing_to_relocate',
 'signal_github_activity_score',
 'signal_search_appearance_30d',
 'signal_saved_by_recruiters_30d',
 'signal_interview_completion_rate',
 'signal_offer_acceptance_rate',
 'signal_verified_email',
 'signal_verified_phone',
 'signal_linkedin_connected',
 'signal_skill_assessment_scores.NLP',
 'signal_skill_assessment_scores.Image Classification',
 'signal_skill_assessment_scores.Fine-tuning LLMs',
 'signal_skill_assessment_scores.Speech Recognition',
 'signal_expected_salary_range_inr_